[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/06_tag_11_12_einfaches_und_gestapeltes_rnn.ipynb)

# Tag 11 & 12 – 06: Einfaches und gestapeltes RNN vergleichen

Beide Modelle erhalten dieselben Zeitfenster und sollen jeweils den nächsten Wert einer periodischen Zeitreihe vorhersagen. Das erste Modell besitzt eine rekurrente Schicht, das zweite zwei rekurrente Schichten. Verglichen werden Modellgröße, Trainingszeit, Loss-Verläufe und die Prognosequalität im späteren Testbereich.

## Daten und Vorhersageaufgabe

Die Zeitreihe verbindet eine schnelle Schwingung, eine langsamere Schwingung und eine langsam veränderliche Amplitude. Aus den letzten 40 Werten soll der nächste Wert vorhergesagt werden.

Die Rohzeitreihe wird chronologisch in Training, Validierung und Test geteilt. Nur die ersten beiden Bereiche beeinflussen das Training; der Testbereich wird erst danach ausgewertet.

In [ ]:
import time                         # Trainingszeit messen
import matplotlib.pyplot as plt     # Zeitreihen- und Loss-Plots zeichnen
import numpy as np                  # Arrays und künstliche Daten erzeugen
import tensorflow as tf             # TensorFlow-RNNs trainieren

plt.style.use('seaborn-v0_8-whitegrid')  # Einheitlicher Plot-Stil
RANDOM_STATE = 13                         # Fester Zufallsstart für reproduzierbare Resultate
np.random.seed(RANDOM_STATE)              # NumPy-Zufallsgenerator festlegen
tf.keras.utils.set_random_seed(RANDOM_STATE)  # TensorFlow-Gewichte reproduzierbar initialisieren

In [ ]:
ANZAHL_WERTE = 600       # Länge der gesamten Zeitreihe
FENSTERLAENGE = 40       # Anzahl vergangener Werte pro Prognose
TRAIN_ENDE = 0.60        # Anteil der Rohzeitreihe für das Training
VALIDIERUNG_ENDE = 0.80  # Anteil der Rohzeitreihe bis zum Ende der Validierung
EPOCHEN = 50             # Maximale Anzahl der Trainingsepochen

zeit = np.arange(ANZAHL_WERTE)  # Zeitindizes erzeugen
rng = np.random.default_rng(RANDOM_STATE)  # Lokalen Zufallsgenerator erzeugen
amplitude = 1.0 + 0.30 * np.sin(2 * np.pi * zeit / 180)  # Langsame Amplitudenänderung erzeugen
werte = (  # Alle Signalanteile zu einer Zeitreihe addieren
    amplitude * np.sin(2 * np.pi * zeit / 30)  # Hauptschwingung mit veränderlicher Amplitude
    + 0.35 * np.sin(2 * np.pi * zeit / 9 + 0.6)  # Schnellere Zusatzschwingung
    + rng.normal(0, 0.08, ANZAHL_WERTE)  # Leichtes Messrauschen
)
train_grenze = int(ANZAHL_WERTE * TRAIN_ENDE)  # Ende des Trainingsbereichs berechnen
validierung_grenze = int(ANZAHL_WERTE * VALIDIERUNG_ENDE)  # Ende des Validierungsbereichs berechnen

mittelwert = werte[:train_grenze].mean()  # Skalierungs-Mittelwert nur aus Trainingsdaten berechnen
standardabweichung = werte[:train_grenze].std()  # Skalierungs-Streuung nur aus Trainingsdaten berechnen
werte_skaliert = (werte - mittelwert) / standardabweichung  # Gesamte Reihe mit Trainingsstatistik normieren

def erstelle_fenster(reihe, fensterlaenge):  # Hilfsfunktion für überlappende Ein-Schritt-Beispiele
    X, y, zielindizes = [], [], []  # Listen für Eingaben, Ziele und Zielpositionen anlegen
    for start in range(len(reihe) - fensterlaenge):  # Jedes mögliche Fenster durchlaufen
        X.append(reihe[start : start + fensterlaenge])  # Vergangene Werte als Modellinput speichern
        y.append(reihe[start + fensterlaenge])  # Direkt folgenden Wert als Ziel speichern
        zielindizes.append(start + fensterlaenge)  # Position des Ziels in der Originalreihe speichern
    return np.asarray(X)[..., np.newaxis], np.asarray(y), np.asarray(zielindizes)  # Zeitachsen-Feature hinzufügen und Arrays zurückgeben

X_alle, y_alle, zielindizes = erstelle_fenster(werte_skaliert, FENSTERLAENGE)  # Fenster aus der skalierten Reihe erzeugen
train_maske = zielindizes < train_grenze  # Ziele vor der ersten Grenze gehören zum Training
val_maske = (zielindizes >= train_grenze) & (zielindizes < validierung_grenze)  # Ziele im mittleren Bereich gehören zur Validierung
test_maske = zielindizes >= validierung_grenze  # Ziele im letzten Bereich gehören zum Test
X_train, y_train = X_alle[train_maske], y_alle[train_maske]  # Trainingsfenster auswählen
X_val, y_val = X_alle[val_maske], y_alle[val_maske]  # Validierungsfenster auswählen
X_test, y_test = X_alle[test_maske], y_alle[test_maske]  # Testfenster auswählen
test_indizes = zielindizes[test_maske]  # Zeitpositionen der Testziele merken

print('Training:', X_train.shape, '| Validierung:', X_val.shape, '| Test:', X_test.shape)  # Größen aller drei Datensätze ausgeben

plt.figure(figsize=(12, 3.8))  # Abbildung für die Zeitreihe anlegen
plt.plot(zeit, werte, color='0.35', linewidth=1.2, label='Zeitreihe')  # Gesamten Signalverlauf zeichnen
plt.axvspan(0, train_grenze - 1, color='tab:blue', alpha=0.10, label='Training')  # Trainingsbereich markieren
plt.axvspan(train_grenze, validierung_grenze - 1, color='tab:orange', alpha=0.10, label='Validierung')  # Validierungsbereich markieren
plt.axvspan(validierung_grenze, ANZAHL_WERTE - 1, color='tab:green', alpha=0.10, label='Test')  # Testbereich markieren
plt.title('Zeitreihe mit chronologischen Datenbereichen')  # Titel setzen
plt.xlabel('Zeitindex')  # x-Achse beschriften
plt.ylabel('Messwert')   # y-Achse beschriften
plt.legend(loc='upper right')  # Legende anzeigen
plt.show()  # Abbildung ausgeben

## Zwei Architekturen

Das einfache RNN gibt nach seiner einzigen rekurrenten Schicht nur den letzten Hidden State an einen Dense-Layer weiter. Beim gestapelten RNN liefert die erste RNN-Schicht dagegen eine vollständige Hidden-State-Sequenz an die zweite RNN-Schicht. Erst die zweite Schicht fasst das Fenster zu einem finalen State zusammen.

In [ ]:
def baue_einfaches_rnn():  # Funktion für das flache Vergleichsmodell definieren
    tf.keras.utils.set_random_seed(RANDOM_STATE)  # Gewichtsinitialisierung reproduzierbar machen
    eingabe = tf.keras.Input(shape=(FENSTERLAENGE, 1), name='zeitfenster')  # 40 Werte mit je einem Merkmal erwarten
    letzter_state = tf.keras.layers.SimpleRNN(16, name='rnn_1')(eingabe)  # Eine RNN-Schicht erzeugt nur ihren letzten State
    ausgabe = tf.keras.layers.Dense(1, name='naechster_wert')(letzter_state)  # Letzten State auf eine Prognose abbilden
    modell = tf.keras.Model(eingabe, ausgabe, name='einfaches_rnn')  # Modell aus Ein- und Ausgabe zusammensetzen
    modell.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse', metrics=['mae'])  # Optimierer und Fehlermaß festlegen
    return modell  # Fertiges Modell zurückgeben

def baue_gestapeltes_rnn():  # Funktion für das tiefe Vergleichsmodell definieren
    tf.keras.utils.set_random_seed(RANDOM_STATE)  # Gewichtsinitialisierung reproduzierbar machen
    eingabe = tf.keras.Input(shape=(FENSTERLAENGE, 1), name='zeitfenster')  # Dieselbe Inputform wie beim einfachen Modell
    erste_folge = tf.keras.layers.SimpleRNN(16, return_sequences=True, name='rnn_1')(eingabe)  # Erste Schicht gibt 40 Hidden States weiter
    letzter_state = tf.keras.layers.SimpleRNN(16, name='rnn_2')(erste_folge)  # Zweite Schicht verdichtet diese Folge zum letzten State
    ausgabe = tf.keras.layers.Dense(1, name='naechster_wert')(letzter_state)  # Letzten State auf eine Prognose abbilden
    modell = tf.keras.Model(eingabe, ausgabe, name='gestapeltes_rnn')  # Modell aus Ein- und Ausgabe zusammensetzen
    modell.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse', metrics=['mae'])  # Optimierer und Fehlermaß festlegen
    return modell  # Fertiges Modell zurückgeben

einfaches_rnn = baue_einfaches_rnn()  # Flaches Modell erzeugen
gestapeltes_rnn = baue_gestapeltes_rnn()  # Tiefes Modell erzeugen

einfaches_rnn.summary()  # Architektur des einfachen Modells ausgeben
gestapeltes_rnn.summary()  # Architektur des gestapelten Modells ausgeben

## Beide Modelle unter gleichen Bedingungen trainieren

Early Stopping beendet einen Trainingslauf, sobald sich der Validierungsfehler mehrere Epochen lang nicht verbessert. Beide Modelle erhalten dieselben Trainings- und Validierungsfenster, dieselbe maximale Epochenzahl und denselben Optimierer.

In [ ]:
def trainiere_und_bewerte(modell):  # Einheitlichen Trainings- und Bewertungsablauf definieren
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)  # Beste Validierungsgewichte behalten
    startzeit = time.perf_counter()  # Startzeit unmittelbar vor dem Training festhalten
    historie = modell.fit(  # Modell auf Trainingsfenstern trainieren
        X_train, y_train,  # Frühere Fenster und ihre Zielwerte
        validation_data=(X_val, y_val),  # Mittleren Zeitbereich nur zur Validierung verwenden
        epochs=EPOCHEN,  # Höchstens die konfigurierte Anzahl Epochen trainieren
        batch_size=32,  # 32 Fenster pro Gewichtsupdate verwenden
        callbacks=[early_stopping],  # Frühzeitiges Stoppen aktivieren
        verbose=0,  # Keine Ausgabe für jede Epoche erzeugen
    )
    trainingszeit = time.perf_counter() - startzeit  # Vergangene Trainingszeit berechnen
    test_prognosen_skaliert = modell.predict(X_test, verbose=0).ravel()  # Skalierte Testprognosen berechnen
    test_prognosen = test_prognosen_skaliert * standardabweichung + mittelwert  # Prognosen zurückskalieren
    test_werte = y_test * standardabweichung + mittelwert  # Testziele zurückskalieren
    test_mae = np.mean(np.abs(test_werte - test_prognosen))  # MAE in Originaleinheit berechnen
    ausgefuehrte_epochen = len(historie.history['loss'])  # Tatsächlich ausgeführte Epochen nach Early Stopping zählen
    return historie, trainingszeit, ausgefuehrte_epochen, test_prognosen, test_mae  # Alle Vergleichswerte zurückgeben

historie_einfach, zeit_einfach, epochen_einfach, prognosen_einfach, mae_einfach = trainiere_und_bewerte(einfaches_rnn)  # Flaches RNN trainieren
historie_tief, zeit_tief, epochen_tief, prognosen_tief, mae_tief = trainiere_und_bewerte(gestapeltes_rnn)  # Gestapeltes RNN trainieren
test_werte = y_test * standardabweichung + mittelwert  # Tatsächliche Testwerte einmal für beide Plots bereitstellen

## Vergleich von Größe, Trainingszeit und Fehlern

Die Tabelle enthält alle Fehler als MAE in der ursprünglichen Einheit der Zeitreihe und erlaubt damit einen direkten Vergleich zwischen Training, Validierung und Test. Die Loss-Grafik zeigt zusätzlich den während des Trainings optimierten MSE-Verlauf. Ein tieferes Modell hat mehr Parameter; ob es im Testbereich tatsächlich besser vorhersagt, zeigt erst die abschließende Test-MAE.

In [ ]:
vergleich = [  # Alle Kennzahlen in einer Liste zusammenfassen
    ('Einfaches RNN', einfaches_rnn.count_params(), epochen_einfach, zeit_einfach, historie_einfach.history['mae'][-1] * standardabweichung, historie_einfach.history['val_mae'][-1] * standardabweichung, mae_einfach),  # Alle MAE-Werte in Originaleinheit
    ('Gestapeltes RNN', gestapeltes_rnn.count_params(), epochen_tief, zeit_tief, historie_tief.history['mae'][-1] * standardabweichung, historie_tief.history['val_mae'][-1] * standardabweichung, mae_tief),  # Alle MAE-Werte in Originaleinheit
]

print(f'{"Modell":<18} {"Parameter":>10} {"Epochen":>8} {"Sek./Ep.":>10} {"Train-MAE":>12} {"Val-MAE":>12} {"Test-MAE":>12}')  # Tabellenkopf ausgeben
print('-' * 100)  # Trennlinie unter dem Tabellenkopf ausgeben
for name, parameter, epochen, sekunden, train_mae, val_mae, test_mae in vergleich:  # Beide Ergebniszeilen nacheinander ausgeben
    print(f'{name:<18} {parameter:>10,} {epochen:>8} {sekunden / epochen:>10.3f} {train_mae:>12.4f} {val_mae:>12.4f} {test_mae:>12.4f}')  # Alle Fehler direkt vergleichbar ausgeben

fig, (ax_loss, ax_mae) = plt.subplots(1, 2, figsize=(13, 4))  # Loss- und Testfehlervergleich nebeneinander anlegen
ax_loss.plot(historie_einfach.history['loss'], color='tab:blue', label='einfach: Training')  # Training-Loss des flachen Modells
ax_loss.plot(historie_einfach.history['val_loss'], color='tab:blue', linestyle='--', label='einfach: Validierung')  # Validierung-Loss des flachen Modells
ax_loss.plot(historie_tief.history['loss'], color='tab:purple', label='gestapelt: Training')  # Training-Loss des tiefen Modells
ax_loss.plot(historie_tief.history['val_loss'], color='tab:purple', linestyle='--', label='gestapelt: Validierung')  # Validierung-Loss des tiefen Modells
ax_loss.set(title='Loss-Verlauf', xlabel='Epoche', ylabel='MSE')  # Linkes Diagramm beschriften
ax_loss.legend()  # Loss-Legende anzeigen

balken = ax_mae.bar(['einfach', 'gestapelt'], [mae_einfach, mae_tief], color=['tab:blue', 'tab:purple'])  # Test-MAEs als Balken zeichnen
ax_mae.bar_label(balken, fmt='%.4f', padding=3)  # Zahlen über den Balken anzeigen
ax_mae.set(title='Fehler im Testbereich', ylabel='Test-MAE')  # Rechtes Diagramm beschriften
plt.tight_layout()  # Abstände zwischen Diagrammen anpassen
plt.show()  # Abbildung ausgeben

## Ein-Schritt-Prognosen im Testbereich

Die beiden Linien zeigen die Prognosen für dieselben späteren Zielwerte. In diesem Vergleich zählt nicht allein der Trainingsfehler, sondern vor allem die Qualität auf den noch nicht verwendeten Testfenstern.

In [ ]:
plt.figure(figsize=(12, 4.5))  # Große Abbildung für Testwerte und beide Prognosekurven anlegen
plt.plot(test_indizes, test_werte, color='tab:green', linewidth=2, label='tatsächliche Testwerte')  # Wahre Zielwerte zeichnen
plt.plot(test_indizes, prognosen_einfach, color='tab:blue', alpha=0.85, label=f'einfaches RNN (MAE={mae_einfach:.3f})')  # Flache RNN-Prognosen zeichnen
plt.plot(test_indizes, prognosen_tief, color='tab:purple', linestyle='--', alpha=0.90, label=f'gestapeltes RNN (MAE={mae_tief:.3f})')  # Tiefe RNN-Prognosen zeichnen
plt.title('Prognosevergleich auf dem späteren Testbereich')  # Titel setzen
plt.xlabel('Zeitindex des vorhergesagten Werts')  # x-Achse beschriften
plt.ylabel('Messwert')  # y-Achse beschriften
plt.legend(loc='upper right')  # Legende anzeigen
plt.show()  # Abbildung ausgeben

## Kerngedanke

Die erste RNN-Schicht eines gestapelten Modells liefert eine zeitliche Folge von Merkmalsdarstellungen an die nächste RNN-Schicht. Dadurch steigt die Zahl der Parameter und die Rechenzeit. Ein Vorteil ist nur dann gegeben, wenn sich der Validierungs- und Testfehler gegenüber dem einfachen Modell tatsächlich verbessert.